# Missão Aurora Siger — Ignition Zero Core

## Sistema de Verificação de Voo Embarcado

> **Contexto:** Na Nova Corrida Espacial, o atraso de comunicação Terra–Marte pode chegar a **22 minutos**. O sistema embarcado deve ser **determinístico e autônomo** durante a janela crítica de decolagem.

---

### Pilares do Sistema

| Pilar | Papel no Sistema |
|---|---|
| Ciência da Computação | Digitalização da realidade física via Thresholding |
| Automação com Python | Lógica de segurança modular e determinística |
| Inteligência Artificial | Análise probabilística e detecção de padrões |
| Energia e Sustentabilidade | Otimização de ciclos de instrução e footprint mínimo |


## 1. Matriz de Variáveis Críticas

| Variável | Descrição Técnica | Tipo | Importância |
|---|---|---|---|
| `temperatura` | Monitoramento térmico de sistemas críticos | `float` (°C) | Estabilidade térmica |
| `integridade` | Extensômetros e sensores ultrassônicos | `bool` (0/1) | Elasticidade do casco |
| `energia` | State of Charge (SoC) das baterias | `%` | Autonomia de suporte à vida |
| `pressao` | Sensores piezoresistivos nos tanques | `float` (PSI) | Previne falhas catastróficas |
| `modulos` | Diagnóstico de prontidão dos módulos | `bool` (flag) | Sistemas secundários ativos |


## 2. Pseudocódigo Estruturado — Arquitetura Fail-Safe

> Filosofia **AND Gate**: a decisão "PRONTO PARA DECOLAR" exige que **todas** as condições sejam verdadeiras simultaneamente. Qualquer anomalia dispara o protocolo de abortagem.

```
INÍCIO
    // Validação de entrada e leitura de sensores
    LER telemetria_bruta
    VALIDAR presença e tipo de todos os sensores obrigatórios
    SE dados inválidos → ABORTAR com log de erro

    // Definição de faixas nominais (Safe Zones)
    CONSTANTE LIMITE_TEMP_MAX  = 35.0   // °C
    CONSTANTE ENERGIA_MINIMA   = 80.0   // %
    CONSTANTE PRESSAO_MIN      = 90.0   // PSI
    CONSTANTE PRESSAO_MAX      = 110.0  // PSI

    // Thresholding: digitalização de sinais analógicos
    temp_ok  ← telemetria.temperatura <= LIMITE_TEMP_MAX
    integ_ok ← telemetria.integridade == 1
    ener_ok  ← telemetria.energia >= ENERGIA_MINIMA
    press_ok ← PRESSAO_MIN <= telemetria.pressao <= PRESSAO_MAX
    mod_ok   ← telemetria.modulos == 1

    // Lógica Gate Strategy (AND Gate — Fail-Safe)
    SE temp_ok E integ_ok E ener_ok E press_ok E mod_ok ENTÃO

        GERAR_LOG "STATUS: PRONTO PARA DECOLAR - SISTEMAS NOMINAIS"
        CALCULAR autonomia_energetica(telemetria.energia)
        INICIAR SEQUÊNCIA_IGNICAO

    SENÃO

        GERAR_LOG "STATUS: DECOLAGEM ABORTADA - FALHA DE SEGURANÇA DETECTADA"
        IDENTIFICAR_ANOMALIA(telemetria, estados)   // caixa-preta auditável
        BLOQUEAR_IGNICAO

    FIM SE
FIM
```

| Variável lógica | Condição de aprovação |
|---|---|
| `temp_ok` | temperatura ≤ 35.0 °C |
| `integ_ok` | integridade == 1 (estrutura nominal) |
| `ener_ok` | energia ≥ 80% (SoC mínimo) |
| `press_ok` | 90.0 ≤ pressão ≤ 110.0 PSI |
| `mod_ok` | módulos == 1 (todos ativos) |

## 2. Importações e Configuração

In [ ]:
from typing import Dict, List, Tuple
from dataclasses import dataclass, field
from datetime import datetime, timezone

# Constantes de Safe Zones (Limiarização)
LIMITE_TEMP_MAX: float = 35.0
ENERGIA_MINIMA: float = 80.0
PRESSAO_MIN: float = 90.0
PRESSAO_MAX: float = 110.0

# Constantes energéticas
CAPACIDADE_TOTAL_KWH: float = 500.0
CONSUMO_DECOLAGEM_KWH: float = 120.0
PERDAS_ENERGETICAS: float = 0.08

print("Configuração carregada com sucesso.")
print(f"  Temp máx.: {LIMITE_TEMP_MAX}°C | Energia mín.: {ENERGIA_MINIMA}% | Pressão: {PRESSAO_MIN}–{PRESSAO_MAX} PSI")

## 3. Estruturas de Dados

In [ ]:
@dataclass
class Telemetria:
    temperatura: float
    integridade: float
    energia: float
    pressao: float
    modulos: float
    timestamp: str = field(default_factory=lambda: datetime.now(timezone.utc).isoformat())

@dataclass
class ResultadoVerificacao:
    pronto: bool
    status: str
    anomalias: List[str]
    timestamp: str
    telemetria: Telemetria
    autonomia_horas: float = 0.0

print("Estruturas de dados definidas.")

## 4. Módulos do Sistema

### 4.1 Validação de Entrada

In [ ]:
SENSORES_OBRIGATORIOS = ["temperatura", "integridade", "energia", "pressao", "modulos"]

def validar_entrada(telemetria: Dict[str, float]) -> Tuple[bool, List[str]]:
    """Valida presença e tipo de todos os campos obrigatórios."""
    erros: List[str] = []
    for sensor in SENSORES_OBRIGATORIOS:
        if sensor not in telemetria:
            erros.append(f"Sensor ausente: '{sensor}'")
        elif not isinstance(telemetria[sensor], (int, float)):
            erros.append(f"Tipo inválido para '{sensor}'")
    return len(erros) == 0, erros

# Teste rápido
ok, errs = validar_entrada({"temperatura": 28.5, "integridade": 1.0,
                             "energia": 95.0, "pressao": 102.3, "modulos": 1.0})
print(f"Entrada válida: {ok} | Erros: {errs}")

ok2, errs2 = validar_entrada({"temperatura": 28.5})
print(f"Entrada inválida: {ok2} | Erros: {errs2}")

### 4.2 Thresholding — Digitalização de Sinais

In [ ]:
def digitalizar_sinais(telemetria: Dict[str, float]) -> Dict[str, bool]:
    """Converte grandezas físicas em estados lógicos booleanos."""
    return {
        "temp_ok":  telemetria["temperatura"] <= LIMITE_TEMP_MAX,
        "integ_ok": telemetria["integridade"] == 1.0,
        "ener_ok":  telemetria["energia"] >= ENERGIA_MINIMA,
        "press_ok": PRESSAO_MIN <= telemetria["pressao"] <= PRESSAO_MAX,
        "mod_ok":   telemetria["modulos"] == 1.0,
    }

exemplo = {"temperatura": 28.5, "integridade": 1.0,
           "energia": 95.0, "pressao": 102.3, "modulos": 1.0}
estados = digitalizar_sinais(exemplo)
for k, v in estados.items():
    print(f"  {k}: {'✓ NOMINAL' if v else '✗ FALHA'}")

### 4.3 Identificação de Anomalias

In [ ]:
def identificar_anomalias(telemetria: Dict[str, float], estados: Dict[str, bool]) -> List[str]:
    """Gera log auditável de anomalias (caixa-preta / black-box)."""
    anomalias: List[str] = []
    if not estados["temp_ok"]:
        anomalias.append(f"TEMPERATURA CRÍTICA: {telemetria['temperatura']}°C (limite: {LIMITE_TEMP_MAX}°C)")
    if not estados["integ_ok"]:
        anomalias.append(f"FALHA DE INTEGRIDADE ESTRUTURAL: flag={telemetria['integridade']}")
    if not estados["ener_ok"]:
        anomalias.append(f"ENERGIA INSUFICIENTE: {telemetria['energia']}% (mínimo: {ENERGIA_MINIMA}%)")
    if not estados["press_ok"]:
        anomalias.append(f"PRESSÃO FORA DO NOMINAL: {telemetria['pressao']} PSI (faixa: {PRESSAO_MIN}–{PRESSAO_MAX} PSI)")
    if not estados["mod_ok"]:
        anomalias.append(f"MÓDULOS INATIVOS: flag={telemetria['modulos']}")
    return anomalias

print("Módulo de identificação de anomalias carregado.")

### 4.4 Análise Energética

**Fórmula de Autonomia:**

$$\text{Autonomia} = \frac{(\text{Capacidade Total} \times \text{Carga Atual}) - \text{Consumo Decolagem}}{\text{Perdas Energéticas}}$$

In [ ]:
def calcular_autonomia(carga_atual_pct: float) -> float:
    """
    Autonomia = ((Capacidade_Total × Carga_Atual) - Consumo_Decolagem)
                / Perdas_Energéticas
    """
    energia_disponivel = CAPACIDADE_TOTAL_KWH * (carga_atual_pct / 100.0)
    energia_util = energia_disponivel - CONSUMO_DECOLAGEM_KWH
    if energia_util <= 0:
        return 0.0
    return round(energia_util / (CAPACIDADE_TOTAL_KWH * PERDAS_ENERGETICAS), 2)

# Demonstração com diferentes SoC
cargas = [60, 70, 80, 90, 95, 100]
print(f"{'SoC (%)':>10} | {'Energia Disponível (kWh)':>25} | {'Autonomia (horas)':>18}")
print("-" * 60)
for c in cargas:
    disp = CAPACIDADE_TOTAL_KWH * (c / 100.0)
    aut = calcular_autonomia(c)
    print(f"{c:>10} | {disp:>25.1f} | {aut:>18.2f}")

### 4.5 Lógica de Decisão Principal — AND Gate (Fail-Safe)

In [ ]:
def validar_sistemas(telemetria_raw: Dict[str, float]) -> ResultadoVerificacao:
    """Verificação determinística completa com filosofia Fail-Safe."""
    timestamp = datetime.now(timezone.utc).isoformat()

    entrada_valida, erros_entrada = validar_entrada(telemetria_raw)
    if not entrada_valida:
        dados = Telemetria(**{k: 0.0 for k in SENSORES_OBRIGATORIOS})
        return ResultadoVerificacao(
            pronto=False, status="ERRO: DADOS INVÁLIDOS",
            anomalias=erros_entrada, timestamp=timestamp,
            telemetria=dados, autonomia_horas=0.0
        )

    dados = Telemetria(
        temperatura=telemetria_raw["temperatura"],
        integridade=telemetria_raw["integridade"],
        energia=telemetria_raw["energia"],
        pressao=telemetria_raw["pressao"],
        modulos=telemetria_raw["modulos"],
        timestamp=timestamp,
    )
    estados = digitalizar_sinais(telemetria_raw)
    anomalias = identificar_anomalias(telemetria_raw, estados)
    pronto = all(estados.values())
    autonomia = calcular_autonomia(telemetria_raw["energia"]) if pronto else 0.0
    status = (
        "STATUS: PRONTO PARA DECOLAR - SISTEMAS NOMINAIS"
        if pronto else
        "STATUS: DECOLAGEM ABORTADA - FALHA DE SEGURANÇA DETECTADA"
    )
    return ResultadoVerificacao(
        pronto=pronto, status=status, anomalias=anomalias,
        timestamp=timestamp, telemetria=dados, autonomia_horas=autonomia
    )

print("Módulo de decisão carregado.")

## 5. Simulações de Cenários

### Cenário 1: Telemetria Nominal (Decolagem Autorizada)

In [ ]:
telemetria_nominal = {
    "temperatura": 28.5,
    "integridade": 1.0,
    "energia": 95.0,
    "pressao": 102.3,
    "modulos": 1.0,
}

r = validar_sistemas(telemetria_nominal)
print(f"Resultado: {r.status}")
print(f"Pronto   : {r.pronto}")
print(f"Autonomia: {r.autonomia_horas} horas")
print(f"Anomalias: {r.anomalias}")

### Cenário 2: Temperatura Crítica

In [ ]:
telemetria_temp_alta = {
    "temperatura": 42.1,   # ACIMA DO LIMITE
    "integridade": 1.0,
    "energia": 92.0,
    "pressao": 100.0,
    "modulos": 1.0,
}

r = validar_sistemas(telemetria_temp_alta)
print(f"Resultado: {r.status}")
for a in r.anomalias:
    print(f"  >> {a}")

### Cenário 3: Múltiplas Falhas Simultâneas

In [ ]:
telemetria_multiplas_falhas = {
    "temperatura": 42.1,   # FALHA: acima do limite
    "integridade": 1.0,
    "energia": 67.5,       # FALHA: abaixo do mínimo
    "pressao": 85.0,       # FALHA: abaixo da faixa
    "modulos": 0.0,        # FALHA: módulos inativos
}

r = validar_sistemas(telemetria_multiplas_falhas)
print(f"Resultado: {r.status}")
print(f"Total de anomalias: {len(r.anomalias)}")
for i, a in enumerate(r.anomalias, 1):
    print(f"  [{i}] {a}")

### Cenário 4: Pressão de Propelente Fora dos Limites

In [ ]:
telemetria_sobrepressao = {
    "temperatura": 30.0,
    "integridade": 1.0,
    "energia": 88.0,
    "pressao": 125.7,      # FALHA: sobrepressão
    "modulos": 1.0,
}

r = validar_sistemas(telemetria_sobrepressao)
print(f"Resultado: {r.status}")
for a in r.anomalias:
    print(f"  >> {a}")

### Cenário 5: Dados Corrompidos (Sensor Ausente)

In [ ]:
telemetria_corrompida = {
    "temperatura": 28.5,
    # 'integridade' ausente — simula falha de sensor/transmissão
    "energia": 91.0,
    "pressao": 101.0,
    "modulos": 1.0,
}

r = validar_sistemas(telemetria_corrompida)
print(f"Resultado: {r.status}")
for a in r.anomalias:
    print(f"  >> {a}")

## 6. Análise Energética Detalhada

In [ ]:
carga = 95.0
energia_disponivel = CAPACIDADE_TOTAL_KWH * (carga / 100.0)
energia_util = energia_disponivel - CONSUMO_DECOLAGEM_KWH
perdas_absolutas = energia_disponivel * PERDAS_ENERGETICAS
autonomia = calcular_autonomia(carga)

print("ANÁLISE ENERGÉTICA — IGNITION ZERO")
print("=" * 45)
print(f"Capacidade Total         : {CAPACIDADE_TOTAL_KWH:.1f} kWh")
print(f"Carga Atual (SoC)        : {carga:.1f}%")
print(f"Energia Disponível       : {energia_disponivel:.1f} kWh")
print(f"Consumo na Decolagem     : {CONSUMO_DECOLAGEM_KWH:.1f} kWh")
print(f"Energia Útil Remanescente: {energia_util:.1f} kWh")
print(f"Perdas Térmicas/Conversão: {perdas_absolutas:.1f} kWh ({PERDAS_ENERGETICAS*100:.0f}%)")
print(f"Autonomia Estimada       : {autonomia:.2f} horas")

## 7. Triagem por IA — Classificação de Risco

A IA atua como camada analítica **probabilística**, complementar à lógica **determinística** do script Python.
Ela **não** autoriza decolagens — identifica tendências e padrões latentes.

In [ ]:
def classificar_risco_ia(telemetria: Dict[str, float]) -> str:
    """
    Simulação de triagem por IA.
    Classifica telemetria em: Nominal, Degradado ou Crítico.
    Em produção, substituir por modelo ML treinado.
    """
    score = 0

    # Temperatura: margem de 5°C abaixo do limite = degradado
    if telemetria["temperatura"] > LIMITE_TEMP_MAX:
        score += 3
    elif telemetria["temperatura"] > LIMITE_TEMP_MAX - 5:
        score += 1

    # Energia: margem de 10% acima do mínimo = degradado
    if telemetria["energia"] < ENERGIA_MINIMA:
        score += 3
    elif telemetria["energia"] < ENERGIA_MINIMA + 10:
        score += 1

    # Pressão: fora da faixa nominal
    if not (PRESSAO_MIN <= telemetria["pressao"] <= PRESSAO_MAX):
        score += 3
    elif not (PRESSAO_MIN + 5 <= telemetria["pressao"] <= PRESSAO_MAX - 5):
        score += 1

    # Integridade e módulos
    if telemetria["integridade"] != 1.0:
        score += 5
    if telemetria["modulos"] != 1.0:
        score += 5

    if score == 0:
        return "NOMINAL"
    elif score <= 3:
        return "DEGRADADO"
    else:
        return "CRÍTICO"

# Classificação dos cenários
cenarios = {
    "Nominal": {"temperatura": 28.5, "integridade": 1.0, "energia": 95.0, "pressao": 102.3, "modulos": 1.0},
    "Temp Alta": {"temperatura": 42.1, "integridade": 1.0, "energia": 92.0, "pressao": 100.0, "modulos": 1.0},
    "Multi-Falha": {"temperatura": 42.1, "integridade": 1.0, "energia": 67.5, "pressao": 85.0, "modulos": 0.0},
    "Energia Baixa": {"temperatura": 29.0, "integridade": 1.0, "energia": 83.0, "pressao": 101.0, "modulos": 1.0},
}

print(f"{'Cenário':<15} | {'Classificação IA':<12} | {'Decisão Determinística'}")
print("-" * 60)
for nome, tel in cenarios.items():
    risco = classificar_risco_ia(tel)
    resultado = validar_sistemas(tel)
    decisao = "DECOLAR" if resultado.pronto else "ABORTAR"
    print(f"{nome:<15} | {risco:<12} | {decisao}")

## 8. Execução Completa — Relatório de Log

In [ ]:
def imprimir_relatorio(resultado: ResultadoVerificacao) -> None:
    sep = "=" * 55
    print(sep)
    print("  AURORA SIGER — IGNITION ZERO CORE")
    print(f"  Timestamp UTC: {resultado.timestamp}")
    print(sep)
    print(f"\n  [RESULTADO]: {resultado.status}\n")
    t = resultado.telemetria
    print("  SENSORES:")
    print(f"    Temperatura : {t.temperatura}°C")
    print(f"    Integridade : {int(t.integridade)}")
    print(f"    Energia     : {t.energia}%")
    print(f"    Pressão     : {t.pressao} PSI")
    print(f"    Módulos     : {int(t.modulos)}")
    if resultado.pronto:
        print(f"\n  Autonomia estimada: {resultado.autonomia_horas} horas")
        print("  >> Iniciando Sequência de Ignição...")
    else:
        print(f"\n  ANOMALIAS ({len(resultado.anomalias)}):")
        for i, a in enumerate(resultado.anomalias, 1):
            print(f"    [{i}] {a}")
        print("  >> Ignição Bloqueada.")
    print(sep)

print("--- CENÁRIO NOMINAL ---")
imprimir_relatorio(validar_sistemas({"temperatura": 28.5, "integridade": 1.0,
                                      "energia": 95.0, "pressao": 102.3, "modulos": 1.0}))

print("\n--- CENÁRIO COM FALHAS ---")
imprimir_relatorio(validar_sistemas({"temperatura": 42.1, "integridade": 1.0,
                                      "energia": 67.5, "pressao": 85.0, "modulos": 1.0}))

---

## 9. Ética, Sustentabilidade e Code Longevity

| Princípio | Implementação no Projeto |
|---|---|
| **Transparência Algorítmica** | Logs auditáveis com identificação precisa de cada anomalia |
| **Determinismo** | AND Gate — ausência de estados ambíguos |
| **Fail-Safe** | Qualquer falha resulta em abortagem (padrão conservador) |
| **Footprint Mínimo** | Sem dependências externas — só biblioteca padrão Python |
| **Code Longevity** | Type Hints, docstrings, modularização e sem efeitos colaterais |
| **Sem Input Interativo** | Dados consumidos via dicionários/streams, não via `input()` |

---

*Missão Aurora Siger — Ignition Zero Core — Sistema desenvolvido para a disciplina de Automação com Python e Fundamentos de IA.*